# NoShowGuard capstone: predicting and explaining missed clinic appointments

Run the cells from top to bottom. The three input files (`appointments.csv`, `clinics.csv`, `followup_notes.csv`) must sit in the same folder as this notebook (in Colab: upload them with the folder icon on the left).

- **Part 1** cleans the data and writes `appointments_clean.csv`, `clinics_enriched.csv`, `rejected_rows.json` and `load_data.sql`.
- **Part 2** is in `queries.sql` (run in MySQL Workbench); this notebook reproduces the same numbers with pandas and SQLite as a cross-check.
- **Part 3** trains and compares the models and writes `test_risk_scores.csv`.
- **Part 4** calls Gemini on the first 10 notes and writes `notes_analysed.csv`. The API key is typed in at run time and is never stored in this file.

# Part 1 — Python & data foundations
## Task 1.1 — Read the file with the csv module

In [ ]:
import csv
import json

with open("appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Rows read:", len(raw_rows))


def parse_row(row):
    """Convert the numeric columns. Raises ValueError if a value cannot be converted."""
    out = dict(row)
    out["age"] = int(row["age"])
    out["fee_inr"] = int(row["fee_inr"])
    out["prior_no_shows"] = int(row["prior_no_shows"])
    out["distance_km"] = float(row["distance_km"])
    return out


accepted, rejected = [], []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted), "| Rejected:", len(rejected))
print("  rejected with blank age        :", sum(1 for r in rejected if r["age"].strip() == ""))
print("  rejected with blank distance_km:", sum(1 for r in rejected if r["distance_km"].strip() == ""))
print("  accepted rows with age > 100 (type check is not a validity check):",
      sum(1 for r in accepted if r["age"] > 100 or r["age"] < 1))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

## Task 1.2 — Inspect and clean, pass 1
This starts again from the original file. The csv-module lists above are not used from here on.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("appointments.csv")
print("Shape:", df.shape)
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Distinct raw booking_channel spellings:", df["booking_channel"].nunique())
print("Rows at start:", len(df))

# (a) drop exact duplicates
df = df.drop_duplicates()
print("After (a) dropping duplicates:", len(df))

# (b) tidy the booking_channel labels
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("After (b) cleaning booking_channel:", len(df), "| labels:", sorted(df["booking_channel"].unique()))

# (c) dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After (c) converting dates:", len(df), "| unreadable appt_date:", df["appt_date"].isna().sum())

# (d) drop rows without an appointment date or a status
n_bad_date = df["appt_date"].isna().sum()
n_bad_status = df["status"].isna().sum()
df = df.dropna(subset=["appt_date", "status"])
print(f"After (d) dropping missing appt_date ({n_bad_date}) or status ({n_bad_status}):", len(df))

## Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
# (a) impossible ages become missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages (below 1 or above 100):", invalid_age.sum())
df.loc[invalid_age, "age"] = np.nan
print("Missing ages in total:", df["age"].isna().sum())

# (b) fill the gaps with medians
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("Median age:", age_median, "| Median distance_km:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) cap outliers at the IQR fences (computed after the gaps are filled)
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_high = (df[col] > upper).sum()
    n_low = (df[col] < lower).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower fence={lower}, upper fence={upper} "
          f"-> capped {n_high + n_low} values ({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee after capping:", df["fee_inr"].max())

## Task 1.4 — Features and a NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean():.1%})")

X_np = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(X_np.mean(axis=0), 2))

# (b) boolean mask for long lead times
mask = X_np[:, 3] > 14
print(f"Rows with lead_days > 14: {mask.sum()} ({mask.mean():.1%})")

# (c) z-scores of distance_km
x = X_np[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

## Task 1.5 — Enrich the clinics from the Open-Meteo API
No key is needed. If a clinic fails twice, its `temp_c` stays empty. No temperature is ever typed in by hand.

In [ ]:
import requests

clinics = pd.read_csv("clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"


def get_temp(lat, lon):
    """Return (temp_c, reason). temp_c is None if the call did not give a usable value."""
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    try:
        resp = requests.get(URL, params=params, timeout=10)
    except requests.exceptions.RequestException as e:
        return None, f"request failed: {e}"
    if resp.status_code != 200:
        return None, f"status {resp.status_code}"
    data = resp.json()
    if "hourly" not in data or not data["hourly"].get("temperature_2m"):
        return None, "hourly data missing in the reply"
    return data["hourly"]["temperature_2m"][0], "ok"


temps = []
for _, c in clinics.iterrows():
    temp, reason = get_temp(c["latitude"], c["longitude"])
    if temp is None:                      # retry this clinic once
        temp, reason = get_temp(c["latitude"], c["longitude"])
    if temp is None:
        print(f"FAILED for {c['clinic_name']}: {reason} -> temp_c left empty")
    else:
        print(f"{c['clinic_name']}: status 200, temp_c = {temp}")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv:", clinics.shape, "| missing temp_c:", clinics["temp_c"].isna().sum())

## Task 1.6 — Export the clean table and write the SQL load script

In [ ]:
COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
        "prior_no_shows", "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[COLS]

print("appointment_id unique:", clean["appointment_id"].is_unique,
      "| distinct values:", clean["appointment_id"].nunique())
print("Missing values across the 16 columns:", int(clean.isnull().sum().sum()))
assert clean["appointment_id"].is_unique and clean.isnull().sum().sum() == 0

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

clinics_e = pd.read_csv("clinics_enriched.csv")
n_inserts = 0
with open("load_data.sql", "w") as f:
    # clinics first: appointments has a foreign key that points to them
    for _, c in clinics_e.iterrows():
        temp_sql = "NULL" if pd.isna(c["temp_c"]) else f"{c['temp_c']}"
        f.write(f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}','{c['city']}',"
                f"'{c['tier']}',{c['latitude']},{c['longitude']},{temp_sql});\n")
        n_inserts += 1
    for _, r in clean.iterrows():
        f.write(f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
                f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},"
                f"{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',"
                f"{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});\n")
        n_inserts += 1
print("Saved load_data.sql with", n_inserts, "INSERT statements")

# Part 2 — SQL cross-check in Python
The graded SQL is in `queries.sql` and is run in MySQL Workbench. This section does two things: the pandas `groupby` check that Task 2.2 asks for, and a run of the same `queries.sql` in SQLite (built into Python) so the SQL numbers can be compared with the Python numbers in one place.

In [ ]:
# Task 2.2 in pandas: clinic counts and no-show counts
appts = pd.read_csv("appointments_clean.csv")
by_clinic = (appts.groupby("clinic_id")
             .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
             .reset_index()
             .merge(clinics_e[["clinic_id", "clinic_name", "city"]], on="clinic_id"))
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)
by_clinic = by_clinic.sort_values("no_show_pct", ascending=False)[
    ["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]]
print(by_clinic.to_string(index=False))

In [ ]:
# Same queries.sql, run in SQLite as a cross-check (MySQL Workbench is still the place to run it for marks)
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
con.executescript("""
CREATE TABLE clinics (
  clinic_id VARCHAR(5) PRIMARY KEY, clinic_name VARCHAR(40) NOT NULL, city VARCHAR(20) NOT NULL,
  tier VARCHAR(10), latitude DECIMAL(8,4), longitude DECIMAL(8,4), temp_c DECIMAL(5,1));
CREATE TABLE appointments (
  appointment_id VARCHAR(8) PRIMARY KEY, patient_id VARCHAR(8) NOT NULL, clinic_id VARCHAR(5) NOT NULL,
  booked_on DATE, appt_date DATE, age INT, gender VARCHAR(1), distance_km DECIMAL(5,1), fee_inr INT,
  booking_channel VARCHAR(10), visit_type VARCHAR(12), reminder_sent VARCHAR(3), prior_no_shows INT,
  status VARCHAR(10), lead_days INT, no_show INT,
  FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id));
""")
con.executescript(open("load_data.sql").read())

sql_text = open("queries.sql").read()
# keep only the SELECT / WITH statements; the CREATE and INSERT lines were handled above
statements = [s.strip() for s in sql_text.split(";") if s.strip()]
for s in statements:
    body = "\n".join(line for line in s.splitlines() if not line.strip().startswith("--")).strip()
    if body.upper().startswith(("SELECT", "WITH")):
        print(pd.read_sql_query(body, con).to_string(index=False), "\n")

try:
    con.execute("INSERT INTO appointments SELECT * FROM appointments WHERE appointment_id = 'A0001'")
except sqlite3.IntegrityError as e:
    print("Duplicate insert of A0001 fails with:", e)

sql_counts = pd.read_sql_query(
    "SELECT c.clinic_name, COUNT(*) AS appointments, SUM(a.no_show) AS no_shows "
    "FROM appointments a JOIN clinics c ON a.clinic_id = c.clinic_id GROUP BY c.clinic_name", con)
check = sql_counts.merge(by_clinic, on="clinic_name", suffixes=("_sql", "_py"))
print("pandas counts match SQL counts:",
      bool((check["appointments_sql"] == check["appointments_py"]).all()
           and (check["no_shows_sql"] == check["no_shows_py"]).all()))

# Part 3 — Machine learning
Rule for this Part: the test set is for reporting only. The model and the threshold are chosen from the training rows.
## Task 3.1 — Features, split and baseline

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score)

data = pd.read_csv("appointments_clean.csv")      # row order is kept exactly as saved
appointment_ids = data["appointment_id"]          # kept aside for Task 3.5(e)

X = data[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent",
          "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = data["no_show"]
print("X columns:", X.shape[1], list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in the test set:", int(y_test.sum()))
baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always predict Attended): {baseline:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

## Task 3.2 — Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)     # fitted on the training set only
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, lr_pred):.3f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", confusion_matrix(y_test, lr_pred))
print(f"Precision: {precision_score(y_test, lr_pred):.3f}")
print(f"Recall   : {recall_score(y_test, lr_pred):.3f}")
print(f"F1       : {f1_score(y_test, lr_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, lr_prob):.3f}")

coefs = pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_[0]})
print("\n", coefs.sort_values("coefficient").to_string(index=False))

i = list(X.columns).index("reminder_sent")
coef_rem = lr.coef_[0][i]
sd_rem = scaler.scale_[i]
print(f"\nreminder_sent coefficient (scaled feature): {coef_rem:.3f}")
print(f"np.exp(coefficient)         = {np.exp(coef_rem):.2f}  (odds multiplier per one standard deviation, SD = {sd_rem:.2f})")
print(f"np.exp(coefficient / scale) = {np.exp(coef_rem / sd_rem):.2f}  (odds multiplier for a real change from No to Yes)")

**Reading the reminder coefficient.** The two numbers differ because the model was fitted on standardised features: `np.exp(coef)` is the change in odds for one standard deviation of `reminder_sent` (about 0.49), whereas going from No to Yes is a full unit, about two standard deviations, so the coefficient has to be divided by the scale first.

In this model, holding the other features constant, appointments with a reminder have odds of a no-show that are roughly 69% lower (odds multiplier about 0.31) than appointments without one. This is an association in observational data, so it does not prove that sending reminders causes fewer no-shows.

## Task 3.3 — Decision tree and overfitting

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, t in [("Full-depth tree", tree_full), ("Depth-3 tree", tree_3)]:
    print(f"{name} (depth {t.get_depth()}): train accuracy {t.score(X_train, y_train):.3f}, "
          f"test accuracy {t.score(X_test, y_test):.3f}")
print(f"Baseline accuracy for comparison: {baseline:.4f}")

The unlimited tree is perfect on the training rows but falls below the always-Attended baseline on the test rows, so the large train-test gap shows it memorised noise (overfitting); the depth-3 tree has a small gap and generalises better.

## Task 3.4 — Random forest and a comparison table

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print(f"Random forest: train accuracy {rf.score(X_train, y_train):.3f}, test accuracy {accuracy_score(y_test, rf_pred):.3f}, "
      f"recall {recall_score(y_test, rf_pred):.3f}, ROC-AUC {roc_auc_score(y_test, rf_prob):.3f}")

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:\n", importances.head(3).round(3).to_string())

tree3_pred = tree_3.predict(X_test)
tree3_prob = tree_3.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, tree3_pred), accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, lr_pred), recall_score(y_test, tree3_pred), recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, lr_prob), roc_auc_score(y_test, tree3_prob), roc_auc_score(y_test, rf_prob)],
}).round(3)
print("\n", comparison.to_string(index=False))

Logistic regression looks best in this table: it has the highest ROC-AUC, so it ranks risky bookings above safe ones better than the other two, and its accuracy and recall are at least as good. This table is for reporting only; the formal choice is made in Task 3.5 from training data.

## Task 3.5 — Tune, choose the model and the threshold, then test once

In [ ]:
# (a) tune the forest on the training set
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print(f"best_score_ (cross-validated ROC-AUC): {grid.best_score_:.3f}")

# (b) choose the model from cross-validated ROC-AUC on the training set only
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_scores = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                                 X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
for name, score in cv_scores.items():
    print(f"{name:32s} CV ROC-AUC = {score:.3f}")
winner = max(cv_scores, key=cv_scores.get)
print("Winner:", winner)
print(f"(For information only) tuned forest on the test set: ROC-AUC = "
      f"{roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]):.3f}")

In [ ]:
# (c) choose the threshold from out-of-fold training probabilities (test set not touched)
assert winner.startswith("Logistic regression"), "The code below assumes logistic regression won; adapt it if not."
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(y_train, flag), "recall": recall_score(y_train, flag),
                 "f1": f1_score(y_train, flag), "flagged": int(flag.sum())})
thr_table = pd.DataFrame(rows).round(3)
print(thr_table.to_string(index=False))
THRESHOLD = float(thr_table.loc[thr_table["f1"].idxmax(), "threshold"])
print("Chosen threshold (higher F1):", THRESHOLD)

The threshold with the higher F1 on the out-of-fold training predictions is chosen. Recall matters more than precision here because the two mistakes do not cost the same: a missed warning leaves an empty slot, while an unnecessary call costs the front desk about a minute.

In [ ]:
# (d) fit the winner on the training set and evaluate once on the test set
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flags = (probs >= THRESHOLD).astype(int)
print(f"Test set at threshold {THRESHOLD}: precision {precision_score(y_test, flags):.3f}, "
      f"recall {recall_score(y_test, flags):.3f}, flagged {int(flags.sum())}, ROC-AUC {roc_auc_score(y_test, probs):.3f}")

# (e) save the risk scores for every test row
scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": flags,
})
scores.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", scores.shape, "| rows with flag = 1:", int(scores["flag"].sum()))

In practice a clinic would set the threshold from how many follow-up calls the front desk can make in an evening, weighed against what an empty slot costs, rather than from F1 alone.

# Part 4 — GenAI: structured reasons from the follow-up notes
## Task 4.1 — One validated call
The key is typed in when the cell runs. It is never written into the notebook. Before uploading to GitHub, clear the output of the cell below if the key is visible in it.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import time
from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = "gemini-3-flash-preview"   # the Flash model from Session 22.1; change it here if your class used another one

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful operations analyst for a chain of outpatient clinics in India. "
               "You read call-centre follow-up notes about missed appointments and return structured JSON only. "
               "You never invent facts that are not in the note.")


def build_prompt_v2(note_text):
    """CRAFT prompt: Context, Role, Action, Tone, Format."""
    return f"""CONTEXT: A clinic chain phones patients who missed an appointment and the caller types a short note. Notes may be in English, Hindi or Hinglish, and may be vague.
ROLE: You are an operations analyst who classifies why the patient did not attend.
ACTION: Read the note and choose exactly one root_cause from this list:
- forgot_or_no_reminder: the patient forgot, mixed up the date, or says no reminder was received
- distance_or_transport: the clinic is too far, or there was no transport or a travel problem
- cost_or_fee: the fee or money was the problem
- illness_or_emergency: the patient or a family member was too unwell, or there was an emergency
- work_or_schedule: work, office, shift, exam or another timing clash
- other: the reason is unclear, missing, or does not fit the list
If the note does not clearly state a reason, answer other rather than guess.
TONE: Neutral and factual, written for a busy clinic manager.
FORMAT: Return JSON with four fields: root_cause (one of the six values, spelled exactly), confidence (a number from 0 to 1), summary (at most 25 words, in English), recommended_action (one short practical step for the front desk).

NOTE: {note_text}"""


def build_prompt_v1(note_text):
    return f"Classify this note: {note_text}"


API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
api_calls = 0     # counts every request sent to Gemini in Tasks 4.1-4.3


def call_gemini(prompt):
    """Send one prompt and return the reply text exactly as the model gave it."""
    global api_calls
    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=2000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))   # if your model rejects this, delete this line
    return response.text


def analyse_note(note_text):
    """Classify one note with the CRAFT prompt and return the parsed dictionary."""
    return json.loads(call_gemini(build_prompt_v2(note_text)))


notes = pd.read_csv("followup_notes.csv")
print("Columns in followup_notes.csv:", list(notes.columns))
NOTE_COL = [c for c in notes.columns if c not in ("note_id", "appointment_id")][-1]   # the text column

n001 = notes.loc[notes["note_id"] == "N001", NOTE_COL].iloc[0]
result_n001 = analyse_note(n001)
print(result_n001)

## Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def is_allowed(root_cause):
    return root_cause in ALLOWED


FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]


def check_reply(raw):
    """Return (fields, status) for one raw reply. Nothing is repaired: missing fields stay empty."""
    fields = {k: None for k in FIELDS}
    try:
        data = json.loads(raw)
        if isinstance(data, dict):
            for k in FIELDS:
                if k in data:
                    fields[k] = data[k]
        FollowupInsight(**data)
        status = "valid" if is_allowed(data["root_cause"]) else "invalid"
    except Exception as e:          # bad JSON, ValidationError, wrong type ...
        print("   invalid reply:", str(e).splitlines()[0])
        status = "invalid"
    return fields, status


records = []
first10 = notes.head(10)
for n, (_, row) in enumerate(first10.iterrows()):
    note_text = row[NOTE_COL]
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"]}

    if pd.isna(note_text) or str(note_text).strip() == "":
        # input validation: no API call for a blank note
        rec.update({"root_cause": "other", "confidence": 0.0, "summary": "No note recorded",
                    "recommended_action": "Re-call patient", "validation_status": "skipped_blank",
                    "raw_response": ""})
        print(row["note_id"], "-> skipped_blank (no API call)")
    else:
        raw = ""
        try:
            raw = call_gemini(build_prompt_v2(note_text))
            fields, status = check_reply(raw)
        except Exception as e:      # network or API error: keep going
            print("   API call failed:", e)
            fields, status = {k: None for k in FIELDS}, "invalid"
        rec.update(fields)
        rec.update({"validation_status": status, "raw_response": raw if raw is not None else ""})
        print(row["note_id"], "->", status, "|", fields["root_cause"])
        if n < len(first10) - 1:
            time.sleep(13)          # stay inside the free-tier rate limit
    records.append(rec)

analysed = pd.DataFrame(records, columns=["note_id", "appointment_id", "root_cause", "confidence", "summary",
                                          "recommended_action", "validation_status", "raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", analysed.shape)
print(analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))

EXPECTED = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N005": "illness_or_emergency", "N006": "work_or_schedule", "N007": "forgot_or_no_reminder",
            "N008": "distance_or_transport", "N009": "other", "N010": "work_or_schedule"}
agree = sum(1 for _, r in analysed.iterrows() if EXPECTED.get(r["note_id"]) == r["root_cause"])
print(f"Agreement with the expected root causes: {agree} of {len(EXPECTED)} non-blank notes")

## Task 4.3 — Break it on purpose

In [ ]:
# (a) two deliberately bad replies
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Patient said the fee was too high.", "recommended_action": "Offer a payment plan"},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Patient said the fee was too high."},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(label, "-> unexpectedly passed")
    except ValidationError as e:
        print(f"{label} -> caught ValidationError:\n{e}\n")

In [ ]:
# (b) V1 (bare prompt) against V2 (CRAFT prompt) on N007 and N009, same schema and settings
rows = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, NOTE_COL].iloc[0]

    time.sleep(13)
    try:
        v1 = json.loads(call_gemini(build_prompt_v1(text)))
        v1_cause = v1.get("root_cause")
    except Exception as e:
        v1_cause = f"(failed: {e})"
    rows.append({"note_id": note_id, "prompt": "V1 bare", "root_cause": v1_cause, "in_allowed_list": is_allowed(v1_cause)})

    # V2 is the CRAFT prompt already used in Task 4.2 at temperature 0, so that result is reused (no extra call)
    v2_cause = analysed.loc[analysed["note_id"] == note_id, "root_cause"].iloc[0]
    rows.append({"note_id": note_id, "prompt": "V2 CRAFT", "root_cause": v2_cause, "in_allowed_list": is_allowed(v2_cause)})

print(pd.DataFrame(rows).to_string(index=False))

# (c) number of API calls
print("\nTotal Gemini API calls across Tasks 4.1-4.3:", api_calls)

I would deploy V2. The bare V1 prompt is free to invent its own labels, so its answers cannot be relied on to match the six allowed values, whereas V2 names the allowed values and tells the model to answer `other` when the note is unclear, which keeps the ambiguous note from being forced into a wrong category.

## Task 4.4 — Root cause against reminder status

In [ ]:
merged = pd.read_csv("notes_analysed.csv").merge(pd.read_csv("appointments_clean.csv"), on="appointment_id")
print("Merged rows:", len(merged))
print(pd.crosstab(merged["root_cause"].fillna("(invalid reply)"), merged["reminder_sent"], margins=True))
forgot = merged[merged["root_cause"] == "forgot_or_no_reminder"]
print(f"\nNotes classed forgot_or_no_reminder: {len(forgot)}, of which reminder_sent = No: {(forgot['reminder_sent'] == 'No').sum()}")
print("Notes whose appointment had reminder_sent = No:", (merged["reminder_sent"] == "No").sum(), "of", len(merged))